In [1]:
# Celula unica - Importacao de dados GPKG para PostgreSQL
import geopandas as gpd
import geoalchemy2
import geopandas as gpd
from sqlalchemy import create_engine
from sqlalchemy import text
from sqlalchemy import create_engine, text
from pathlib import Path

# =============
# CONFIGURACOES 
# =============

# Pasta onde estao os arquivos GPKG
PASTA_DADOS = Path(r"C:\Temp")  #ALTERAR PARA SUA PASTA. EX: "C:\Downloads"

# Nomes dos arquivos
CAR_ARQUIVO = "es_mt_car_20260406"
SIGEF_ARQUIVO = "pa_br_sigef_privado_20260107"

# Nomes das tabelas no banco
CAR_TABELA = "es_mt_car_20260406"
SIGEF_TABELA = "pa_br_sigef_privado_20260107"

# Controle de importacao (True = importa, False = pula)
IMPORTAR_CAR = True
IMPORTAR_SIGEF = True

# Se False, mantem a tabela existente no banco (nao reimporta)
RECRIAR_TABELAS = False

# Credenciais do banco local
DB_HOST = "localhost"
DB_PORT = "5432"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_NAME = "geotec"

# ==========================================
# FUNCOES
# ==========================================

def conectar_banco():
    return create_engine(
        f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    )


def criar_schemas(engine):
    with engine.connect() as conn:
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS car"))
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS incra"))
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))
        conn.commit()
    print("[OK] Schemas car e incra prontos")


def tabela_existe(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            conn.execute(text(f"SELECT 1 FROM {schema}.{tabela} LIMIT 1"))
            return True
    except Exception:
        return False


def contar_registros(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            r = conn.execute(text(f"SELECT COUNT(*) FROM {schema}.{tabela}"))
            return r.fetchone()[0]
    except Exception:
        return 0


def encontrar_arquivo(pasta, nome_arquivo):
    """Tenta localizar o arquivo: .gpkg -> .shp"""
    nome_base = Path(nome_arquivo).stem
    
    # Tenta GPKG primeiro
    gpkg_path = pasta / f"{nome_base}.gpkg"
    if gpkg_path.exists():
        return gpkg_path, 'gpkg'
    
    # Tenta SHP
    shp_path = pasta / f"{nome_base}.shp"
    if shp_path.exists():
        return shp_path, 'shp'
    
    return None, None


def importar_arquivo(engine, caminho, tipo_arquivo, schema, tabela):
    """Importa arquivo (GPKG ou SHP) para o PostgreSQL"""
    print(f"  Lendo: {caminho.name} (formato: {tipo_arquivo.upper()})")
    
    try:
        gdf = gpd.read_file(str(caminho))
    except Exception as e:
        print(f"  ERRO ao ler arquivo: {e}")
        return False
    
    print(f"  Registros: {len(gdf)}")

    tem_geometria = (
        isinstance(gdf, gpd.GeoDataFrame)
        and gdf.active_geometry_name is not None
        and not gdf.geometry.isna().all()
    )

    if tem_geometria:
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4674)
        else:
            gdf = gdf.to_crs(epsg=4674)
        gdf.to_postgis(
            name=tabela, con=engine, schema=schema,
            if_exists="replace", index=False
        )
    else:
        df = pd.DataFrame(gdf)
        geom_col = getattr(gdf, "active_geometry_name", None)
        if geom_col and geom_col in df.columns:
            df = df.drop(columns=geom_col)
        print("  [INFO] Sem geometria ativa; importando como tabela comum")
        df.to_sql(
            name=tabela, con=engine, schema=schema,
            if_exists="replace", index=False
        )

    print(f"  OK -> {schema}.{tabela}")
    return True


def importar_se_necessario(engine, nome_arquivo, schema, tabela, ativo):
    if not ativo:
        print(f"  [SKIP] {schema}.{tabela} (desativado)")
        return

    caminho, tipo_arquivo = encontrar_arquivo(PASTA_DADOS, nome_arquivo)
    
    if caminho is None:
        print(f"  [ERRO] Arquivo nao encontrado: {nome_arquivo} (GPKG ou SHP) em {PASTA_DADOS}")
        return

    if tabela_existe(engine, schema, tabela) and not RECRIAR_TABELAS:
        n = contar_registros(engine, schema, tabela)
        print(f"  [EXISTE] {schema}.{tabela} ({n} registros) - mantido")
        return

    importar_arquivo(engine, caminho, tipo_arquivo, schema, tabela)

# ==========================================
# MAIN
# ==========================================

def main():
    print("=" * 60)
    print("IMPORTACAO DE DADOS - AULA 07")
    print("=" * 60)

    if not PASTA_DADOS.exists():
        print(f"[ERRO] Pasta nao encontrada: {PASTA_DADOS}")
        return

    engine = conectar_banco()
    print("[OK] Conectado ao PostgreSQL")

    criar_schemas(engine)

    # Importar CAR
    print("\n[1] Importando CAR")
    importar_se_necessario(engine, CAR_ARQUIVO, "car", CAR_TABELA, IMPORTAR_CAR)

    # Importar SIGEF
    print("\n[2] Importando SIGEF")
    importar_se_necessario(engine, SIGEF_ARQUIVO, "incra", SIGEF_TABELA, IMPORTAR_SIGEF)

    # Resumo final
    print("\n" + "=" * 60)
    print("RESUMO DAS TABELAS")
    print("=" * 60)
    print(f"  car.{CAR_TABELA}: {contar_registros(engine, 'car', CAR_TABELA)} registros")
    print(f"  incra.{SIGEF_TABELA}: {contar_registros(engine, 'incra', SIGEF_TABELA)} registros")

    engine.dispose()
    print("\n[OK] Importacao concluida!")
    print("=" * 60)


if __name__ == "__main__":
    main()

IMPORTACAO DE DADOS - AULA 07
[OK] Conectado ao PostgreSQL
[OK] Schemas car e incra prontos

[1] Importando CAR
  [EXISTE] car.es_mt_car_20260406 (213912 registros) - mantido

[2] Importando SIGEF
  [EXISTE] incra.pa_br_sigef_privado_20260107 (1498605 registros) - mantido

RESUMO DAS TABELAS
  car.es_mt_car_20260406: 213912 registros
  incra.pa_br_sigef_privado_20260107: 1498605 registros

[OK] Importacao concluida!
